In [1]:
!pip install -q telebot
!pip install -q pydub

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.1/48.1 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 270.5/270.5 kB 5.3 MB/s eta 0:00:00


In [9]:
import librosa
import numpy as np
import tensorflow as tf

from pydub import AudioSegment
import telebot

In [10]:
mybot = telebot.TeleBot("8117558510:AAEeNCga7moOCAPL-PUbQtwrFxv2aQelDJI", parse_mode=None)
labels = ['abdollah', 'azra', 'davood', 'javad', 'khadijeh', 'kiana', 'mahdi', 'maryam', 'matin', 'melika', 'mohadeseh', 'mohammad', 'mohammad_parvari', 'mona', 'nahid', 'nima', 'omid', 'parisa', 'parsa', 'sajedeh', 'shima', 'tara']
my_model = tf.keras.models.load_model('/content/drive/MyDrive/Colab_Notebooks/PyLearn7-Assignment59/AudioClassification/weights/audio_classification.h5')


In [12]:
@mybot.message_handler(commands=['start'])
def send_welcome(message):
    msg = mybot.send_message(message.chat.id,"Hi "+str(message.chat.first_name) + "\n" +
                             "Welcome to 'pydio classifier bot'" + "\n" +
                             "If your name is in the list below, this bot can recognize your sound." + "\n" +
                             "[Abdollah, Azra, Davood, Javad, Khadijeh, Kiana, Mahdi, Maryam,Matin, Melika, Mohadeseh, Mohammad, Mohammad_parvari, Mona, Nahid,Nima, Omid, Parisa, Parsa, Sajedeh, Shima, Tara]" + "\n" +
                             "/start_audio_recognition")


@mybot.message_handler(commands=['start_audio_recognition'])
def send_audio(message):
    msg = mybot.reply_to(message,"Please send me a few seconds of your voice.")
    mybot.register_next_step_handler(msg,recognize_audio)


def recognize_audio(message):

    fileID = message.voice.file_id
    file_info = mybot.get_file(fileID)
    file_path = file_info.file_path
    downloaded_file = mybot.download_file(file_path)

    with open(file_path, 'wb') as new_file:
        new_file.write(downloaded_file)

    wav, _ = librosa.load(file_path, sr=None, mono=True, duration=30, offset=0.0)

    length = 48000
    resized_wavefrom = librosa.util.fix_length(wav, size=length)

    input_data = np.expand_dims(resized_wavefrom, axis=-1)
    input_data = np.expand_dims(resized_wavefrom, axis=0)

    pred = my_model.predict(input_data)
    predicted_class = np.argmax(pred)
    label = labels[predicted_class]

    print("The voice belongs to: ", label)
    mybot.reply_to(message, f'The voice belongs to {label}')
    # mybot.send_message(message.chat.id, "The voice belongs to: ", label)

# mybot.enable_save_next_step_handlers(delay=2)
# mybot.load_next_step_handlers()
mybot.infinity_polling()

2025-04-10 20:15:56,123 (__init__.py:1123 MainThread) ERROR - TeleBot: "Break infinity polling"
ERROR:TeleBot:Break infinity polling
